# Обучение модели ResNet18 для классификации AI vs Human Generated Images

Этот notebook содержит код для обучения модели ResNet18 на датасете Train_1 и оценки качества на Test_1.

## 1. Импорт библиотек

In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 2. Подготовка данных

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.root_dir / self.data.iloc[idx]['file_name']
        image = Image.open(img_path).convert('RGB')
        label = self.data.iloc[idx]['label']

        if self.transform:
            image = self.transform(image)

        return image, label

In [27]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageDataset(
    csv_file='drive/MyDrive/DL/LD/ai-vs-human-generated-dataset-hw/Train_1/train.csv',
    root_dir='drive/MyDrive/DL/LD/ai-vs-human-generated-dataset-hw/Train_1',
    transform=train_transform
)

test_dataset = ImageDataset(
    csv_file='drive/MyDrive/DL/LD/ai-vs-human-generated-dataset-hw/Test_1/test.csv',
    root_dir='drive/MyDrive/DL/LD/ai-vs-human-generated-dataset-hw/Test_1',
    transform=test_transform
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

Train dataset size: 9993
Test dataset size: 3997


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 3. Создание модели ResNet18

In [28]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

print(f'Model architecture:')
print(model)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model architecture:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): R

## 4. Определение функции потерь и оптимизатора

In [29]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## 5. Функция обучения

In [30]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(dataloader, desc='Training'):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')

    return epoch_loss, epoch_acc, epoch_f1

## 6. Функция валидации

In [31]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    epoch_precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    epoch_recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)

    return epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

## 7. Обучение модели

In [41]:
from torch.utils.tensorboard import SummaryWriter
import datetime

num_epochs = 10
train_losses = []
train_accs = []
train_f1s = []

batch_size = 32

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_dir = f"my_logs/run_{timestamp}"
writer = SummaryWriter(log_dir=log_dir)

writer.add_text("hyperparameters/batch_size", str(batch_size))
writer.add_text("hyperparameters/learning_rate", "0.001")
writer.add_text("hyperparameters/num_epochs", str(num_epochs))
writer.add_text("hyperparameters/optimizer", "Adam")
writer.add_text("hyperparameters/scheduler", "StepLR(step_size=5, gamma=0.1)")

writer.add_graph(model, torch.randn(1, 3, 224, 224).to(device))

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')

    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
    scheduler.step()

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_f1s.append(train_f1)

    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_acc, epoch)
    writer.add_scalar('F1/train', train_f1, epoch)
    writer.add_scalar('Learning_rate', scheduler.get_last_lr()[0], epoch)

    print(f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')

print('\nTraining completed!')
writer.close()


Epoch 1/10


Training:   0%|          | 0/313 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Training: 100%|██████████| 313/313 [02:11<00:00,  2.38it/s]


Train Loss: 0.0658, Acc: 0.9750, F1: 0.9750

Epoch 2/10


Training:   0%|          | 0/313 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Training: 100%|██████████| 313/313 [02:01<00:00,  2.58it/s]


Train Loss: 0.0680, Acc: 0.9734, F1: 0.9734

Epoch 3/10


...

Train Loss: 0.0577, Acc: 0.9799, F1: 0.9799

Epoch 9/10


Training:   0%|          | 0/313 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Training: 100%|██████████| 313/313 [01:53<00:00,  2.75it/s]


Train Loss: 0.0532, Acc: 0.9799, F1: 0.9799

Epoch 10/10


Training:   0%|          | 0/313 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Training: 100%|██████████| 313/313 [02:00<00:00,  2.59it/s]

Train Loss: 0.0550, Acc: 0.9789, F1: 0.9789

Training completed!


## 8. Оценка модели на тестовом датасете

In [33]:
from datetime import datetime

test_log_dir = f"my_logs/test_{timestamp}"
test_writer = SummaryWriter(log_dir=test_log_dir)

test_loss, test_acc, test_f1, test_precision, test_recall = validate(model, test_loader, criterion, device)

test_writer.add_scalar('Metrics/loss', test_loss, 0)
test_writer.add_scalar('Metrics/accuracy', test_acc, 0)
test_writer.add_scalar('Metrics/f1_score', test_f1, 0)
test_writer.add_scalar('Metrics/precision', test_precision, 0)
test_writer.add_scalar('Metrics/recall', test_recall, 0)

test_writer.add_text('Test_Report/full', f"""
Test Results on Test_1:
- Loss: {test_loss:.4f}
- Accuracy: {test_acc:.4f}
- F1 Score: {test_f1:.4f}
- Precision: {test_precision:.4f}
- Recall: {test_recall:.4f}
""")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")

test_writer.close()

Validation:   0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Validation: 100%|██████████| 125/125 [00:36<00:00,  3.47it/s]

Test Loss: 0.0950
Test Accuracy: 0.9667
Test F1 Score: 0.9667
Test Precision: 0.9669
Test Recall: 0.9667


In [37]:
import os

os.makedirs("models", exist_ok=True)
model_path = "models/resnet18_base_model.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'train_accs': train_accs,
    'train_f1s': train_f1s,
    'test_metrics': {
        'loss': test_loss,
        'accuracy': test_acc,
        'f1': test_f1,
        'precision': test_precision,
        'recall': test_recall
    }
}, model_path)

In [38]:
train2_dataset = ImageDataset(
    csv_file='drive/MyDrive/DL/LD/ai-vs-human-generated-dataset-hw/Train_2/train.csv',
    root_dir='drive/MyDrive/DL/LD/ai-vs-human-generated-dataset-hw/Train_2',
    transform=train_transform
)

test2_dataset = ImageDataset(
    csv_file='drive/MyDrive/DL/LD/ai-vs-human-generated-dataset-hw/Test_2/test.csv',
    root_dir='drive/MyDrive/DL/LD/ai-vs-human-generated-dataset-hw/Test_2',
    transform=test_transform
)

train2_loader = DataLoader(train2_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test2_loader = DataLoader(test2_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f'Train_2 dataset size: {len(train2_dataset)}')
print(f'Test_2 dataset size: {len(test2_dataset)}')

Train_2 dataset size: 3997
Test_2 dataset size: 2000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [42]:
finetune_model = models.resnet18(pretrained=False)
num_features = finetune_model.fc.in_features
finetune_model.fc = nn.Linear(num_features, 2)
checkpoint = torch.load("models/resnet18_base_model.pth")
finetune_model.load_state_dict(checkpoint['model_state_dict'])
finetune_model = finetune_model.to(device)

finetune_optimizer = optim.Adam(finetune_model.parameters(), lr=0.0001)
finetune_scheduler = optim.lr_scheduler.StepLR(finetune_optimizer, step_size=5, gamma=0.1)

num_finetune_epochs = 5
finetune_losses = []
finetune_accs = []
finetune_f1s = []

finetune_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
finetune_log_dir = f"my_logs/finetune_{finetune_timestamp}"
finetune_writer = SummaryWriter(log_dir=finetune_log_dir)

finetune_writer.add_text("finetune_params/learning_rate", "0.0001")
finetune_writer.add_text("finetune_params/epochs", str(num_finetune_epochs))
finetune_writer.add_text("finetune_params/pretrained_model", "resnet18_base_model.pth")

for epoch in range(num_finetune_epochs):
    print(f'\n Epoch {epoch+1}/{num_finetune_epochs}')

    train_loss, train_acc, train_f1 = train_epoch(finetune_model, train2_loader, criterion, finetune_optimizer, device)
    finetune_scheduler.step()

    finetune_losses.append(train_loss)
    finetune_accs.append(train_acc)
    finetune_f1s.append(train_f1)

    finetune_writer.add_scalar('Loss/finetune_train', train_loss, epoch)
    finetune_writer.add_scalar('Accuracy/finetune_train', train_acc, epoch)
    finetune_writer.add_scalar('F1/finetune_train', train_f1, epoch)

    print(f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')

finetune_writer.close()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)



 Epoch 1/5


Training:   0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Training: 100%|██████████| 125/125 [06:02<00:00,  2.90s/it]


Train Loss: 0.1236, Acc: 0.9547, F1: 0.9547

 Epoch 2/5


Training:   0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Training: 100%|██████████| 125/125 [00:46<00:00,  2.68it/s]


...

Train Loss: 0.1002, Acc: 0.9617, F1: 0.9617

 Epoch 4/5


Training:   0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Training: 100%|██████████| 125/125 [00:48<00:00,  2.57it/s]


Train Loss: 0.0869, Acc: 0.9657, F1: 0.9657

 Epoch 5/5


Training:   0%|          | 0/125 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Training: 100%|██████████| 125/125 [00:52<00:00,  2.40it/s]

Train Loss: 0.0901, Acc: 0.9627, F1: 0.9627


In [44]:
test2_loss, test2_acc, test2_f1, test2_precision, test2_recall = validate(finetune_model, test2_loader, criterion, device)

print(f"Test Loss: {test2_loss:.4f}")
print(f"Test Accuracy: {test2_acc:.4f}")
print(f"Test F1 Score: {test2_f1:.4f}")
print(f"Test Precision: {test2_precision:.4f}")
print(f"Test Recall: {test2_recall:.4f}")

test2_writer = SummaryWriter(log_dir=f"my_logs/finetune_test_{finetune_timestamp}")
test2_writer.add_scalar('Metrics/test_loss', test2_loss, 0)
test2_writer.add_scalar('Metrics/test_accuracy', test2_acc, 0)
test2_writer.add_scalar('Metrics/test_f1_score', test2_f1, 0)
test2_writer.add_scalar('Metrics/test_precision', test2_precision, 0)
test2_writer.add_scalar('Metrics/test_recall', test2_recall, 0)
test2_writer.close()

Validation: 100%|██████████| 63/63 [03:08<00:00,  2.99s/it]

Test Loss: 0.0644
Test Accuracy: 0.9805
Test F1 Score: 0.9805
Test Precision: 0.9805
Test Recall: 0.9805
